# 04 · Interaktive Erklärung gespeicherter Fälle

Zeigt für ein Beispiel: die Köpfe beider Modelle, die Zerlegung, die Credal-Intervalle und die gewählte Reaktion.
Keine neuen API-Kosten. Vor der finalen Evaluation werden nur Dev-Beispiele angeboten.
Die menschlichen Stimmen werden nur zum Vergleich gezeigt; die Reaktion wird ohne sie berechnet.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config/project.json").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import display, Image
from uncertainty_project.common import config
CFG = config(ROOT)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
print("Projekt:", ROOT.name, "| Modelle:", list(CFG["models"]))

In [ ]:
import numpy as np
from uncertainty_project.api import model_keys, read_cached, experiment_directory
from uncertainty_project.data import load_split
from uncertainty_project.experiment import load_bundle
from uncertainty_project.features import feature_frame, entropy
from uncertainty_project.models import predict_assessors, run_policies
from uncertainty_project.common import LABELS

KEYS = model_keys(ROOT)
try:
    bundles = {k: load_bundle(ROOT, k) for k in KEYS}
    evaluated = (experiment_directory(ROOT) / "evaluation_record.json").exists()
    split_name = "test" if evaluated else "dev"
    examples = load_split(ROOT, split_name, include_test_labels=evaluated)
    DEMO_READY = True
    print("Demo-Datenteil:", split_name)
except FileNotFoundError as exc:
    DEMO_READY = False
    print(exc)

In [ ]:
TEXT = {"answer": "Antwort direkt geben",
        "hedge": "Antwort mit ausdrücklicher Unsicherheit / Enthaltung",
        "alternatives": "Mehrere legitime Lesarten zeigen",
        "clarify": "Rückfrage stellen (Bedeutung oder Bezug unklar)"}

def show_case(uid, model_key):
    selected = examples.loc[examples.uid == uid].copy()
    row = selected.iloc[0]
    features = feature_frame(ROOT, selected, model_key)
    bundle = bundles[model_key]
    pred = predict_assessors(bundle, features)
    actions, top2 = run_policies(bundle, features, pred)["decomposed"]
    f = features.iloc[0]
    print("Ausgangssatz:", row.premise)
    print("Hypothese:   ", row.hypothesis)
    heads = [read_cached(ROOT, row.to_dict(), v, model_key)["parsed"] for v in range(5)]
    display(pd.DataFrame(heads)[list(LABELS) + ["lexical_ambiguity", "unclear_reference", "probabilistic_inference"]].round(2)
            .rename(index=lambda i: f"Kopf {i+1}"))
    print(f"Modellantwort ({model_key}):", LABELS[int(f.base_label)])
    print("Zerlegung  - erwartete Entropie (aleatorisch, Modellsicht):", round(f.expected_entropy, 3),
          "| Uneinigkeit der Köpfe (epistemisch):", round(f.epistemic_mi, 3))
    print("Credal     - Intervallbreite:", round(f.credal_width, 3), "| eindeutig dominierendes Label:", bool(f.credal_determinate))
    print("Bewerter   - vorhergesagte menschliche Uneinigkeit:", round(float(entropy(pred["distributions"]["tree_full"])[0]), 3),
          "| epistemisches Fehlerrisiko:", round(float(pred["risks"]["epistemic_tree_full"][0]), 3),
          "| klärbarer Hinweis:", round(f.clarifiable_cue, 2))
    print("\nREAKTION:", actions[0], "->", TEXT[actions[0]])
    if actions[0] in ("alternatives", "clarify"):
        print("Angebotene Lesarten:", [LABELS[i] for i in top2[0]])
    print("Schwellen:", bundle["policies"]["decomposed"])
    display(pd.DataFrame({"Label": LABELS,
                          "Vorhergesagter menschlicher Anteil": pred["distributions"]["tree_full"][0].round(3),
                          "Beobachteter Anteil (nur Vergleich)": np.round(row.human_distribution, 3)}))

In [ ]:
if DEMO_READY:
    try:
        import ipywidgets as widgets
        widgets.interact(show_case, uid=widgets.Dropdown(options=examples.uid.tolist()[:300]),
                         model_key=widgets.ToggleButtons(options=KEYS))
    except ImportError:
        show_case(examples.uid.iloc[0], KEYS[0])

**Bezug zur Masterarbeit:** Die Reaktionen `alternatives` und `clarify` entsprechen der strukturierten Darstellung
mehrerer Perspektiven aus der Thesis. Dieses Projekt ergänzt die vorgelagerte Entscheidung, *wann* diese Darstellung angebracht ist.
Für die Abschlussdemo kann `show_case` in die bestehende Weboberfläche übertragen werden.